# 02. Model Training & Evaluation — Vehicle Casting Defect Detection

> **Runtime Environment:** Google Colab (GPU: T4 or better recommended)
> **Target Output:** `models/model.keras`, confusion matrix, and training curves in `reports/figures/`

## 0. Google Colab Environment Setup
Run this section if running in **Google Colab**. It clones/pulls the repository, installs pinned dependencies, downloads the Kaggle dataset, and verifies the directory layout.

In [ ]:
# Step 0a: Clone or pull the repository
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_dir = 'vehicle-defect-detection'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/annugupta25/vehicle-defect-detection.git
        %cd vehicle-defect-detection
    else:
        %cd vehicle-defect-detection
        !git pull origin main
    
    if '.' not in sys.path:
        sys.path.insert(0, '.')
    print("Current working directory:", os.getcwd())
else:
    print("Running locally. Project root confirmed.")

In [ ]:
# Step 0b: Install pinned dependencies from requirements.txt
if IN_COLAB:
    !pip install -q -r requirements.txt
    print("Dependencies installed successfully!")
else:
    print("Local environment: Ensure requirements.txt is installed.")

In [ ]:
# Step 0c: Kaggle authentication and dataset download
dataset_target = os.path.join("data", "casting_data", "casting_data")

if not os.path.exists(dataset_target):
    print("Dataset not found. Initiating download...")
    if IN_COLAB:
        from google.colab import files
        if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
            print("Please upload your 'kaggle.json' API token file:")
            uploaded = files.upload()
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
        
        !kaggle datasets download -d ravirajsinh45/real-life-industrial-dataset-of-casting-product
        !mkdir -p data
        !unzip -q -o real-life-industrial-dataset-of-casting-product.zip -d data/
        print("Dataset extracted to data/")
    else:
        print("Please place the unzipped dataset in 'data/casting_data/casting_data/' locally.")
else:
    print(f"Dataset already present at: {dataset_target}")

In [ ]:
# Step 0d: Verify folder layout with src/utils.py
from src.utils import check_dataset_paths

status = check_dataset_paths("data/casting_data/casting_data")
print("Dataset path check results:")
for k, v in status.items():
    print(f"  - {k}: {v}")

assert status["train_exists"] and status["test_exists"], (
    "Dataset validation failed! Please verify the folder layout before proceeding."
)
print("\n Dataset layout verified! Ready to proceed.")

## 1. Hardware Verification

In [ ]:
import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f" GPU Active: {gpus}")
else:
    print("⚠️ No GPU detected. In Colab, go to: Runtime -> Change runtime type -> Select T4 GPU.")

## 2. Load Datasets & Apply Augmentation Pipeline

Using `src.data_loader.load_datasets` with 85/15 train/val split (seed=42) and augmentation applied strictly to the training pipeline.

In [ ]:
from src.data_loader import load_datasets, get_augmentation_pipeline, configure_performance

DATA_DIR = "data/casting_data/casting_data"

# Load datasets (224x224, batch 32, 85/15 train/val split)
train_ds, val_ds, test_ds, class_names = load_datasets(
    data_dir=DATA_DIR,
    img_size=(224, 224),
    batch_size=32,
    validation_split=0.15,
    seed=42,
    color_mode="rgb",
)
print(f"Classes: {class_names}")

# Configure performance: augmentation strictly on train set
aug_pipeline = get_augmentation_pipeline()
train_ds_perf = configure_performance(train_ds, is_training=True, augment=True, augmentation_pipeline=aug_pipeline)
val_ds_perf = configure_performance(val_ds, is_training=False, augment=False)
test_ds_perf = configure_performance(test_ds, is_training=False, augment=False)
print("Data pipelines ready for training.")

## 3. Build & Compile Model

Using `src.model.build_transfer_model` with MobileNetV2 pretrained backbone (or `build_baseline_cnn`).

In [ ]:
from src.model import build_transfer_model, build_baseline_cnn, compile_model

# Build MobileNetV2 transfer model (or use build_baseline_cnn())
model = build_transfer_model(input_shape=(224, 224, 3), backbone_name="MobileNetV2", dropout_rate=0.3)
model = compile_model(model, learning_rate=1e-4)
model.summary()

## 4. Train Model with Callbacks

In [ ]:
from tensorflow import keras

os.makedirs("models", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath="models/model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        verbose=1,
    ),
]

EPOCHS = 20
history = model.fit(
    train_ds_perf,
    validation_data=val_ds_perf,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 5. Holdout Test Evaluation & Grad-CAM Verification

Evaluating strictly on `test_ds_perf` from the holdout `test/` directory.

In [ ]:
from src.evaluate import evaluate_dataset, plot_confusion_matrix, plot_training_history
from src.gradcam import find_last_conv_layer, make_gradcam_heatmap, overlay_gradcam

# Evaluate on holdout test set
results = evaluate_dataset(model, test_ds_perf, threshold=0.5)
print("Holdout Test Set Results:")
print(f"  F1 Score: {results['f1_score']:.4f}")
print(f"  ROC-AUC:  {results['roc_auc']:.4f}")

# Save and display confusion matrix
cm_fig = plot_confusion_matrix(
    results["y_true"],
    results["y_pred"],
    class_names=["Defective (def_front)", "OK (ok_front)"],
    save_path="reports/figures/confusion_matrix.png",
)
plt.show()

# Save and display training curves
hist_fig = plot_training_history(
    history,
    save_path="reports/figures/training_history.png",
)
plt.show()

# Test Grad-CAM on one test batch
for test_images, test_labels in test_ds_perf.take(1):
    sample_img = test_images[0:1]
    sample_label = int(test_labels[0].numpy()[0])
    last_conv = find_last_conv_layer(model)
    heatmap = make_gradcam_heatmap(sample_img, model, last_conv_layer_name=last_conv)
    overlay, _ = overlay_gradcam(sample_img[0].numpy().astype('uint8'), heatmap)
    
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(sample_img[0].numpy().astype('uint8'))
    plt.title(f"Test Sample (True: {class_names[sample_label]})")
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title("Grad-CAM Heatmap Overlay")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig("reports/figures/gradcam_sample.png", dpi=300)
    plt.show()
    break

## 6. Artifact Export & Model Download

Confirms `models/model.keras`, confusion matrix, and training curves in `reports/figures/`, and downloads `model.keras` for local Streamlit deployment.

In [ ]:
import os

model_file = "models/model.keras"
assert os.path.exists(model_file), f"Model file not found at {model_file}!"
size_mb = os.path.getsize(model_file) / (1024 * 1024)

print(f"✓ Model saved at: {model_file} ({size_mb:.2f} MB)")
print(f"✓ Confusion Matrix: reports/figures/confusion_matrix.png")
print(f"✓ Training History: reports/figures/training_history.png")

if IN_COLAB:
    from google.colab import files
    print("Initiating download of model.keras to your local computer...")
    files.download(model_file)
    print("Download started! Place the file in vehicle-defect-detection/models/model.keras")
else:
    print("Running locally: model is already placed at models/model.keras")